# GOES / STIX X-ray QPP residuals vs the EUV wave kinematics

To isolate quasi-periodic pulsations, smooth each thermal X-ray light curve with a
slow trend $\tilde F$ and subtract it: $r(t)=F(t)-\tilde F(t)$. A Savitzky–Golay
filter whose window is several times the QPP period works as the trend. The residual
is then overlaid on the wave kinematics (loaded from the CSV exported by
`plot_SECCHI_extended.ipynb`) so the QPP phase can be matched to the front.

## 0. Imports and parameters

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import logging
import sunpy
sunpy.log.setLevel(logging.WARNING)

import numpy as np
import pandas as pd
import astropy.units as u
from sunpy.net import Fido, attrs as a
import sunpy.timeseries as ts
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

date = '2025-10-06'


## 1. Load and detrend the X-ray light curves

In [ ]:
def load_goes_xrs(t0, t1):
    """Fetch GOES/XRS flux as a TimeSeries over [t0, t1] (same style as solar_data_utils)."""
    res = Fido.search(a.Time(t0, t1), a.Instrument('XRS'))
    files = Fido.fetch(res)
    g = ts.TimeSeries(files, concatenate=True)
    return g.truncate(t0, t1)


def qpp_residual(time, flux, window_s=None, n_window=None, polyorder=3):
    """Detrend a light curve; return (trend, residual).

    Smoothing window set by window_s (seconds, via median cadence) or n_window
    (odd sample count).
    """
    flux = np.asarray(flux, dtype=float)
    t = pd.to_datetime(time)
    if n_window is None:
        cad = np.median(np.diff(t).astype('timedelta64[s]').astype(float))
        n_window = max(5, int(round(window_s / cad)))
    if n_window % 2 == 0:
        n_window += 1
    n_window = min(n_window, len(flux) - (1 - len(flux) % 2))
    trend = savgol_filter(flux, n_window, polyorder, mode='interp')
    return trend, flux - trend


In [ ]:
goes = load_goes_xrs(f'{date}T08:30:00', f'{date}T09:20:00')
gdf  = goes.to_dataframe()
g_t  = gdf.index
g_flux = gdf['xrsb'].values            # 1-8 A long (thermal) channel
g_trend, g_res = qpp_residual(g_t, g_flux, window_s=120, polyorder=3)


In [ ]:
# STIX thermal time series: load via stixpy if installed, else plug your own arrays.
try:
    from stixpy.product import Product
    # lc = Product('/path/to/solo_L1_stix-ql-lightcurve_*.fits')
    # stix_t, stix_flux = lc.time.datetime, lc.data['counts'][:, 0]
    raise ImportError  # remove once you wire a real file path
except Exception:
    stix_t = pd.date_range(f'{date}T08:30:00', f'{date}T09:20:00', freq='4s')
    stix_flux = np.full(len(stix_t), np.nan)

if np.isfinite(stix_flux).any():
    s_trend, s_res = qpp_residual(stix_t, stix_flux, window_s=120, polyorder=3)
else:
    s_trend = s_res = None


## 2. Overlay residuals on the wave kinematics

In [ ]:
# Kinematics CSV exported from plot_SECCHI_extended.ipynb (edit the filename/columns).
kin_csv = './slit_03_front_00_EUVI_kinematics.csv'
try:
    dk = pd.read_csv(kin_csv, parse_dates=['time'])
    k_time = dk['time']
    k_dist = dk['distance_mean_Mm']
    k_speed = dk['speed_mean_km/s']
except FileNotFoundError:
    k_time = k_dist = k_speed = None
    print(f'{kin_csv} not found -- run the J-map kinematics first or fix the path.')


In [ ]:
fig, axs = plt.subplots(3, 1, figsize=[9, 10], sharex=True)

axs[0].plot(g_t, g_flux, lw=1, label='GOES 1-8 Å')
axs[0].plot(g_t, g_trend, lw=1.5, color='k', alpha=0.7, label='trend')
axs[0].set_yscale('log'); axs[0].set_ylabel('Flux [W m$^{-2}$]'); axs[0].legend(fontsize=8)

axs[1].plot(g_t, g_res, lw=1, color='tab:blue', label='GOES residual')
if s_res is not None:
    axr = axs[1].twinx()
    axr.plot(stix_t, s_res, lw=1, color='tab:orange', label='STIX residual')
    axr.set_ylabel('STIX residual', color='tab:orange')
axs[1].axhline(0, color='k', lw=0.8, alpha=0.5)
axs[1].set_ylabel('GOES residual', color='tab:blue'); axs[1].legend(fontsize=8, loc='upper left')

if k_time is not None:
    axs[2].plot(k_time, k_dist, 'o-', ms=3, color='tab:green', label='front distance')
    ax2b = axs[2].twinx()
    ax2b.plot(k_time, k_speed, 's--', ms=3, color='tab:purple', alpha=0.7, label='front speed')
    axs[2].set_ylabel('Distance [Mm]', color='tab:green')
    ax2b.set_ylabel('Speed [km/s]', color='tab:purple')
axs[2].set_xlabel('Time [UT]')
axs[2].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
fig.autofmt_xdate(); fig.suptitle('X-ray QPP residuals vs EUV wave kinematics')
fig.tight_layout(); plt.show()
